# Pipeline walkthrough — trace one or more samples end to end

**Not a graded artifact** (see `01`/`02`/`03` for those) — this is a debugging/understanding notebook: pick any request, run it through the real pipeline exactly once (same `run_pipeline()` the API and `recommend()` use — no reimplementation), and print what happened at every stage:

1. **Context** assembled (occasion/formality/weather/wardrobe)
2. **Router + queries** built (which KB layers get hit, and with what query text)
3. **Retrieval** — what was actually pulled from the KB, per layer
4. **Generation** — the raw structured LLM output (items + rationale + citations)
5. **Cited result** — the final human-readable outfit + resolved sources

> Cross-reference: LangSmith (`whattowear-rag` project) shows the same trace as a proper span tree with timing/token counts — this notebook is the quick, readable, no-context-switch version.

In [1]:
from whattowear.kb import get_kb
from whattowear.pipeline.run import run_pipeline
from whattowear.pipeline import query_builder as qb, cite

kb = get_kb()   # reconnects to the persistent local store if already built, embeds once if not
print('collection:', kb.collection, '| chunks:', len(kb.chunks))

INFO    whattowear.ingest.build_kb: source: Wikipedia: Color theory                              layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 48 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Wikipedia: Color harmony                             layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 12 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Wikipedia: Complementary colors                      layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 38 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Chevreul: Principles of Harmony & Contrast of Colours layer=L1  loader=epub      status=have
INFO    whattowear.ingest.build_kb:   -> 45 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Munsell: A Color Notation                            layer=L1  loader=epub      status=have
INFO    whattowear.ingest.build_kb:   -> 39 chunk(s) [section]
INFO

collection: whattowear_kb | chunks: 391


### `trace()` — one real pipeline call, then print every stage

Calls `run_pipeline()` **once** (the actual production entrypoint — same cost as a single `recommend()` call, nothing duplicated), then re-derives the router/query-builder view for display (free, pure functions, no extra API calls) and prints each stage's real output.

In [2]:
def trace(occasion, **kwargs):
    strategy = kwargs.get('strategy', 'advanced')
    header = ', '.join(f'{k}={v!r}' for k, v in kwargs.items())
    print(f"\n{'='*78}\nREQUEST: occasion={occasion!r}" + (f', {header}' if header else '') + f"\n{'='*78}")

    run = run_pipeline(occasion, **kwargs)   # the ONE real call -- exactly what recommend() does

    ctx = run.ctx
    print('\n--- Stage 1: Context assembled ---')
    print(f'  occasion={ctx.occasion}  formality={ctx.formality}  mood={ctx.mood}')
    print(f'  temp_c={ctx.temp_c}  temp_band={ctx.temp_band}  season={ctx.season}  condition={ctx.condition}')
    print(f'  wardrobe: {len(ctx.wardrobe)} item(s)')

    layers = qb.route(ctx)   # free, pure -- recomputed only for display
    print('\n--- Stage 2: Router + queries built ---')
    print(f'  layers routed: {layers}   (L2 must never appear here)')
    print(f'  naive_query (baseline):  {qb.naive_query(ctx)!r}')
    print(f'  l3_query (trend search): {qb.l3_query(ctx)!r}')

    print(f'\n--- Stage 3: Retrieved ({run.retrieval.strategy}) ---')
    for name, docs in [('L4', run.retrieval.l4), ('L1', run.retrieval.l1), ('L3', run.retrieval.l3)]:
        print(f'  {name} ({len(docs)}):')
        for d in docs:
            preview = d.page_content.strip().replace(chr(10), ' ')[:80]
            print(f'    [{d.metadata["rule_id"]}] {preview}')

    print('\n--- Stage 4: Raw generation (structured LLM output) ---')
    for i, outfit in enumerate(run.generation.outfits, 1):
        print(f'  Outfit {i}: {outfit.items}')
        for r in outfit.rationale:
            print(f'    - {r.text}  [cites: {r.cites}]')

    print('\n--- Stage 5: Cited result ---')
    print(cite.render_text(run.result))
    return run

### One sample

In [4]:
_ = trace('wedding', mood='elegant', temp_c=12, strategy='advanced')


REQUEST: occasion='wedding', mood='elegant', temp_c=12, strategy='advanced'


INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/embeddings "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://0815614a-f654-42cb-ae37-4a5eb6e1fc9f.eu-west-1-0.aws.cloud.qdrant.io:6333/collections/whattowear_kb/points/query "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK"



--- Stage 1: Context assembled ---
  occasion=wedding  formality=formal  mood=elegant
  temp_c=12.0  temp_band=cool  season=summer  condition=None
  wardrobe: 40 item(s)

--- Stage 2: Router + queries built ---
  layers routed: ['L4', 'L1', 'L3']   (L2 must never appear here)
  naive_query (baseline):  'what to wear for wedding, formal, elegant mood, cool weather, summer'
  l3_query (trend search): 'summer wedding outfit formal elegant tone cool weather'

--- Stage 3: Retrieved (advanced) ---
  L4 (3):
    [L4-dc-formal] Formal: a dark, well-tailored suit or a floor-length or elegant evening dress. M
    [L4-occ-wedding-evening] Evening wedding eligibility: formal to black-tie depending on the invitation. Fa
    [L4-wx-cool] Cool weather (10-16C): a light jacket, cardigan, or blazer over a top; layering 
  L1 (15):
    [L1-color-complementary] Complementary pairing: colors opposite on the wheel (e.g. blue/orange, red/green
    [L1-color-analogous] Analogous pairing: colors adjacent on

### Multiple samples — see how the flow adapts

A deliberately varied set: a cold office request, a hot beach request, and the wedding case above again but through `hybrid` instead of `advanced`, so you can compare retrieval directly.

In [5]:
samples = [
    dict(occasion='office', formality='business_casual', temp_c=8, strategy='advanced'),
    dict(occasion='beach', mood='relaxed', temp_c=30, strategy='advanced'),
    dict(occasion='wedding', mood='elegant', temp_c=12, strategy='hybrid'),
]
for s in samples:
    trace(s.pop('occasion'), **s)


REQUEST: occasion='office', formality='business_casual', temp_c=8, strategy='advanced'


INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/embeddings "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://0815614a-f654-42cb-ae37-4a5eb6e1fc9f.eu-west-1-0.aws.cloud.qdrant.io:6333/collections/whattowear_kb/points/query "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK"



--- Stage 1: Context assembled ---
  occasion=office  formality=business_casual  mood=None
  temp_c=8.0  temp_band=cold  season=summer  condition=None
  wardrobe: 40 item(s)

--- Stage 2: Router + queries built ---
  layers routed: ['L4', 'L1', 'L3']   (L2 must never appear here)
  naive_query (baseline):  'what to wear for office, business casual, cold weather, summer'
  l3_query (trend search): 'summer office outfit business casual cold weather'

--- Stage 3: Retrieved (advanced) ---
  L4 (3):
    [L4-dc-businesscasual] Business casual: professional without a suit. Trousers or a knee-length skirt wi
    [L4-occ-office] Office / work eligibility: default to business casual unless the workplace is ex
    [L4-wx-cold] Cold weather (0-10C): a warm outer layer over a mid-layer (sweater or jacket). L
  L1 (15):
    [L1-color-complementary] Complementary pairing: colors opposite on the wheel (e.g. blue/orange, red/green
    [L1-color-analogous] Analogous pairing: colors adjacent on the whe

INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/embeddings "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://0815614a-f654-42cb-ae37-4a5eb6e1fc9f.eu-west-1-0.aws.cloud.qdrant.io:6333/collections/whattowear_kb/points/query "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK"



--- Stage 1: Context assembled ---
  occasion=beach  formality=casual  mood=relaxed
  temp_c=30.0  temp_band=hot  season=summer  condition=None
  wardrobe: 40 item(s)

--- Stage 2: Router + queries built ---
  layers routed: ['L4', 'L1', 'L3']   (L2 must never appear here)
  naive_query (baseline):  'what to wear for beach, casual, relaxed mood, hot weather, summer'
  l3_query (trend search): 'summer beach outfit casual relaxed tone hot weather'

--- Stage 3: Retrieved (advanced) ---
  L4 (3):
    [L4-dc-casual] Casual dress code: everyday comfort-first clothing. Jeans, chinos, T-shirts, cas
    [L4-occ-beach] Beach / seaside eligibility: casual and heat-appropriate. Light, breathable fabr
    [L4-wx-hot] Hot weather (above 28C): minimal, loose, light-colored breathable clothing; prio
  L1 (15):
    [L1-color-complementary] Complementary pairing: colors opposite on the wheel (e.g. blue/orange, red/green
    [L1-color-analogous] Analogous pairing: colors adjacent on the wheel (e.g. blu

INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/embeddings "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://0815614a-f654-42cb-ae37-4a5eb6e1fc9f.eu-west-1-0.aws.cloud.qdrant.io:6333/collections/whattowear_kb/points/query "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK"



--- Stage 1: Context assembled ---
  occasion=wedding  formality=formal  mood=elegant
  temp_c=12.0  temp_band=cool  season=summer  condition=None
  wardrobe: 40 item(s)

--- Stage 2: Router + queries built ---
  layers routed: ['L4', 'L1', 'L3']   (L2 must never appear here)
  naive_query (baseline):  'what to wear for wedding, formal, elegant mood, cool weather, summer'
  l3_query (trend search): 'summer wedding outfit formal elegant tone cool weather'

--- Stage 3: Retrieved (hybrid) ---
  L4 (3):
    [L4-dc-formal] Formal: a dark, well-tailored suit or a floor-length or elegant evening dress. M
    [L4-occ-wedding-evening] Evening wedding eligibility: formal to black-tie depending on the invitation. Fa
    [L4-wx-cool] Cool weather (10-16C): a light jacket, cardigan, or blazer over a top; layering 
  L1 (15):
    [L1-color-complementary] Complementary pairing: colors opposite on the wheel (e.g. blue/orange, red/green
    [L1-color-analogous] Analogous pairing: colors adjacent on t

### One more — a genuinely new case

Different occasion, formality, mood, and temperature band from everything above (`date` / `smart_casual` / `mild`, vs. the `wedding`/`office`/`beach` and `formal`/`business_casual`/`casual` combinations already covered) — same wardrobe fixture, just confirming the pipeline (and the new Qdrant Cloud store) handles a fresh combination correctly.

In [6]:
_ = trace('date', mood='romantic', temp_c=18, strategy='advanced')


REQUEST: occasion='date', mood='romantic', temp_c=18, strategy='advanced'


INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/embeddings "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://0815614a-f654-42cb-ae37-4a5eb6e1fc9f.eu-west-1-0.aws.cloud.qdrant.io:6333/collections/whattowear_kb/points/query "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"
INFO    httpx: HTTP Request: POST https://ai-gateway.vercel.sh/v1/chat/completions "HTTP/1.1 200 OK"



--- Stage 1: Context assembled ---
  occasion=date  formality=smart_casual  mood=romantic
  temp_c=18.0  temp_band=mild  season=summer  condition=None
  wardrobe: 40 item(s)

--- Stage 2: Router + queries built ---
  layers routed: ['L4', 'L1', 'L3']   (L2 must never appear here)
  naive_query (baseline):  'what to wear for date, smart casual, romantic mood, mild weather, summer'
  l3_query (trend search): 'summer date outfit smart casual romantic tone mild weather'

--- Stage 3: Retrieved (advanced) ---
  L4 (3):
    [L4-dc-smartcasual] Smart casual: polished but relaxed. Combine a casual base with one elevated piec
    [L4-occ-date] Dinner date eligibility: smart casual is the reliable default; nudge toward the 
    [L4-wx-mild] Mild weather (16-22C): a single comfortable layer with an optional light topper 
  L1 (15):
    [L1-color-complementary] Complementary pairing: colors opposite on the wheel (e.g. blue/orange, red/green
    [L1-color-analogous] Analogous pairing: colors adjac